# Qwen3 + Two-Tower Model for T2DM Comorbidity PATHWAY Prediction
## ECE176 Final Project — Colab (GPU Required)

**Task:** Given a T2DM patient's diagnosis history (ICD code sequence + demographics), predict the **NEXT comorbidity** they will develop.
This is a **multi-class classification** problem with **26 comorbidity classes**.

**Architecture:**
- **Tower A (ICD Embedding):** Patient ICD code sequences → Qwen3 embedding extraction
- **Tower B (Structured Features):** 38 features (10 covariates + 26 comorbidity flags + 2 temporal)
- **Fusion:** Concatenate → MLP → 26-class Softmax output

**Data from local package:**
- `pathway_data.pkl` — Patient pathways, feature names, label encoder
- `pathway_samples.pkl` — Train/test transition samples
- `patient_icd_sequences.csv` — Time-ordered ICD sequences per patient
- `comorbidity_definitions.json` — Comorbidity ICD code definitions
- `pathway_baseline_summary.json` — Baseline model results for comparison

## 0. Environment Setup

In [ ]:
!pip install -q transformers accelerate torch bitsandbytes scipy

import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ★ 修改這裡指向你上傳的資料夾路徑
DATA_DIR = '/content/drive/MyDrive/colab_qwen3_pathway_package'  # <-- 改成你的路徑

import os, json, pickle
import pandas as pd
import numpy as np

# Verify files exist
for f in ['pathway_data.pkl', 'pathway_samples.pkl', 'patient_icd_sequences.csv',
          'comorbidity_definitions.json']:
    path = os.path.join(DATA_DIR, f)
    exists = os.path.exists(path)
    size = os.path.getsize(path) / 1e6 if exists else 0
    print(f"  {'\u2713' if exists else '\u2717'} {f:45s} {size:.1f} MB")

## 1. Load Data

In [ ]:
# Load pathway data
with open(os.path.join(DATA_DIR, 'pathway_data.pkl'), 'rb') as f:
    pathway_data = pickle.load(f)

# Load train/test samples
with open(os.path.join(DATA_DIR, 'pathway_samples.pkl'), 'rb') as f:
    pathway_samples = pickle.load(f)

train_samples = pathway_samples['train']
test_samples = pathway_samples['test']

# Load ICD sequences
icd_seq_df = pd.read_csv(os.path.join(DATA_DIR, 'patient_icd_sequences.csv'))

# Load comorbidity definitions
with open(os.path.join(DATA_DIR, 'comorbidity_definitions.json'), 'r') as f:
    definitions = json.load(f)

# Extract metadata
COMORBIDITIES = pathway_data['label_encoder_classes']
FEATURE_NAMES = pathway_data['feature_names']
COVARIATES = pathway_data['covariates']
N_CLASSES = pathway_data['n_classes']
patient_pathways = pathway_data['patient_pathways']

# Load baseline results for comparison
baseline_path = os.path.join(DATA_DIR, 'pathway_baseline_summary.json')
if os.path.exists(baseline_path):
    with open(baseline_path, 'r') as f:
        baseline_results = json.load(f)
    print('Baseline results loaded for comparison')
else:
    baseline_results = None
    print('No baseline results found')

print(f'\nComorbidity classes: {N_CLASSES}')
print(f'Feature dimension: {len(FEATURE_NAMES)}')
print(f'Train samples: {len(train_samples):,}')
print(f'Test samples: {len(test_samples):,}')
print(f'Patients with ICD sequences: {len(icd_seq_df):,}')
print(f'\nComorbidities: {COMORBIDITIES}')

## 2. Load Qwen3 Model

Using Qwen3-1.7B with 4-bit quantization. Extract the **last hidden state** as patient embeddings from ICD code sequences.

> If VRAM is limited (T4 = 16GB), use 4-bit quantization.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

# ★ 選擇模型大小
MODEL_NAME = "Qwen/Qwen3-1.7B"  # 可換成 "Qwen/Qwen3-0.6B"

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

EMBED_DIM = model.config.hidden_size
print(f"Model loaded! Hidden size: {EMBED_DIM}")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

## 3. Build Prompt & Extract Embeddings

For each patient, construct a clinical prompt from their ICD sequence and extract Qwen3's last hidden state.
The prompt is specifically designed for **pathway prediction** — asking the model to predict the next comorbidity.

In [ ]:
def build_pathway_prompt(icd_sequence_str, seen_comorbidities=None, max_codes=80):
    """
    將 ICD 序列轉為 Qwen3 可讀的臨床摘要 prompt，專為路徑預測設計。
    """
    entries = icd_sequence_str.split('|')
    if len(entries) > max_codes:
        entries = entries[-max_codes:]

    lines = []
    for entry in entries:
        try:
            ver_code, days = entry.split('@')
            ver, code = ver_code.split(':')
            ver_label = 'ICD-10' if ver == 'V10' else 'ICD-9'
            lines.append(f"  Day {days}: {code} ({ver_label})")
        except:
            continue

    prompt = (
        "Patient diagnosed with Type 2 Diabetes Mellitus (T2DM). "
        "Below is their medical history as ICD diagnosis codes in chronological order "
        "(Day 0 = T2DM diagnosis date, negative days = before T2DM):\n\n"
        + "\n".join(lines)
    )

    if seen_comorbidities:
        prompt += f"\n\nComorbidities already developed: {', '.join(seen_comorbidities)}"

    prompt += (
        "\n\nBased on the diagnosis history above, "
        "predict the next most likely comorbidity this patient will develop."
    )
    return prompt


# Test with one patient
test_sid = list(patient_pathways.keys())[0]
test_seq = icd_seq_df[icd_seq_df['subject_id'] == test_sid]['icd_sequence'].values[0]
test_pathway = patient_pathways[test_sid]
example_prompt = build_pathway_prompt(test_seq, seen_comorbidities=[p[0] for p in test_pathway[:2]])
print(f"Prompt length: {len(example_prompt)} chars")
print(f"Token count: {len(tokenizer.encode(example_prompt))}")
print(f"\n--- Example Prompt (first 600 chars) ---")
print(example_prompt[:600])

In [ ]:
@torch.no_grad()
def extract_embedding(prompt, tokenizer, model, max_length=512):
    """
    從 Qwen3 提取最後一層 hidden state 的 mean pooling 作為 embedding。
    """
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
        padding=False,
    ).to(model.device)

    outputs = model(**inputs, output_hidden_states=True)
    last_hidden = outputs.hidden_states[-1]  # (1, seq_len, hidden_dim)
    attention_mask = inputs['attention_mask'].unsqueeze(-1)
    masked = last_hidden * attention_mask
    embedding = masked.sum(dim=1) / attention_mask.sum(dim=1)  # (1, hidden_dim)

    return embedding.squeeze(0).cpu().float().numpy()


# Test
test_emb = extract_embedding(example_prompt, tokenizer, model)
print(f"Embedding shape: {test_emb.shape}")
print(f"Embedding stats: mean={test_emb.mean():.4f}, std={test_emb.std():.4f}")
print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
import time

# ============================================================
# Batch extract embeddings for ALL patients
# ★ T4 GPU: ~0.3-0.5 sec/patient -> 3-4 hours for all
# ★ Set RUN_ALL = True to process all patients
# ============================================================

icd_map = icd_seq_df.set_index('subject_id')['icd_sequence'].to_dict()

# Collect all patient IDs that need embeddings
all_patient_ids = set()
for s in train_samples + test_samples:
    all_patient_ids.add(s['subject_id'])
all_patient_ids = sorted(all_patient_ids & set(icd_map.keys()))

print(f"Total patients to embed: {len(all_patient_ids):,}")

# ★ Set to True to run all, False for subset testing
RUN_ALL = False
if not RUN_ALL:
    SUBSET_SIZE = 500
    all_patient_ids = all_patient_ids[:SUBSET_SIZE]
    print(f"  -> Running SUBSET mode: {SUBSET_SIZE} patients")

EMBED_CACHE = os.path.join(DATA_DIR, 'qwen3_pathway_embeddings.pkl')
if os.path.exists(EMBED_CACHE) and RUN_ALL:
    print("Loading cached embeddings...")
    with open(EMBED_CACHE, 'rb') as f:
        patient_embeddings = pickle.load(f)
    print(f"  Loaded {len(patient_embeddings)} embeddings")
else:
    patient_embeddings = {}
    start = time.time()

    for i, sid in enumerate(all_patient_ids):
        if sid in patient_embeddings:
            continue

        seq = icd_map.get(sid, "")
        if not seq:
            continue

        # Build pathway-specific prompt
        seen = []
        if sid in patient_pathways:
            seen = [p[0] for p in patient_pathways[sid][:3]]  # First few comorbidities
        prompt = build_pathway_prompt(seq, seen_comorbidities=seen if seen else None)
        emb = extract_embedding(prompt, tokenizer, model)
        patient_embeddings[sid] = emb

        if (i + 1) % 50 == 0:
            elapsed = time.time() - start
            rate = (i + 1) / elapsed
            remaining = (len(all_patient_ids) - i - 1) / rate
            print(f"  [{i+1}/{len(all_patient_ids)}] "
                  f"{rate:.1f} patients/sec, "
                  f"ETA: {remaining/60:.0f} min")

    elapsed = time.time() - start
    print(f"\nDone! {len(patient_embeddings)} embeddings in {elapsed/60:.1f} min")

    with open(EMBED_CACHE, 'wb') as f:
        pickle.dump(patient_embeddings, f)
    print(f"Cached to {EMBED_CACHE}")

## 4. Pathway Prediction Model (PyTorch)

**Architecture:**
```
Tower A: Qwen3 Embedding (hidden_dim) → Linear(256) → ReLU → Dropout
Tower B: 38 features                  → Linear(64)  → ReLU → Dropout
                    ↓ concat (320) ↓
            Fusion: Linear(128) → ReLU → Dropout
                    Linear(26)  → Softmax (26 comorbidity classes)
```

Ablation: compare **with** vs **without** Qwen3 embeddings.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, top_k_accuracy_score

# ============================================================
# Dataset
# ============================================================
class PathwayDataset(Dataset):
    """
    Each sample: (qwen_embedding, structured_features, label)
    label is an integer 0-25 representing the next comorbidity.
    """
    def __init__(self, samples, patient_embeddings, embed_dim):
        self.data = []
        for s in samples:
            sid = s['subject_id']
            features = s['features']  # 38-dim
            label = s['label']  # int 0-25
            emb = patient_embeddings.get(sid, np.zeros(embed_dim))
            self.data.append((emb, features, label))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        emb, feats, label = self.data[idx]
        return (torch.tensor(emb, dtype=torch.float32),
                torch.tensor(feats, dtype=torch.float32),
                torch.tensor(label, dtype=torch.long))


# ============================================================
# Two-Tower Model (Pathway)
# ============================================================
class PathwayTwoTower(nn.Module):
    """
    Tower A: Qwen3 embedding -> compress to 256-d
    Tower B: 38 structured features -> expand to 64-d
    Fusion: concat(256 + 64) -> 128 -> 26 classes
    """
    def __init__(self, embed_dim, n_features, n_classes, use_qwen=True):
        super().__init__()
        self.use_qwen = use_qwen

        self.tower_a = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

        self.tower_b = nn.Sequential(
            nn.Linear(n_features, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
        )

        fusion_in = 256 + 64 if use_qwen else 64
        self.fusion = nn.Sequential(
            nn.Linear(fusion_in, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, n_classes),  # 26-class output
        )

    def forward(self, qwen_emb, structured):
        feat_b = self.tower_b(structured)
        if self.use_qwen:
            feat_a = self.tower_a(qwen_emb)
            fused = torch.cat([feat_a, feat_b], dim=1)
        else:
            fused = feat_b
        return self.fusion(fused)  # logits, no softmax (use CrossEntropyLoss)


# ============================================================
# Tabular ResNet (Ablation: structured-only with skip connections)
# ============================================================
class PathwayResNet(nn.Module):
    def __init__(self, n_features, n_classes, hidden_dim=128, n_blocks=3):
        super().__init__()
        self.input_proj = nn.Linear(n_features, hidden_dim)
        self.blocks = nn.ModuleList()
        for _ in range(n_blocks):
            self.blocks.append(nn.Sequential(
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(hidden_dim, hidden_dim),
            ))
        self.head = nn.Sequential(
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_classes),
        )

    def forward(self, qwen_emb, structured):
        x = self.input_proj(structured)
        for block in self.blocks:
            x = x + block(x)
        return self.head(x)


print("Models defined: PathwayTwoTower, PathwayResNet")

## 5. Training Loop

In [ ]:
def evaluate_pathway(model, loader, device, n_classes):
    """Evaluate pathway prediction metrics."""
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for emb, feats, labels in loader:
            emb, feats = emb.to(device), feats.to(device)
            logits = model(emb, feats)
            probs = torch.softmax(logits, dim=1)
            all_probs.append(probs.cpu())
            all_labels.append(labels)

    all_probs = torch.cat(all_probs).numpy()
    all_labels = torch.cat(all_labels).numpy()

    results = {}
    for k in [1, 3, 5]:
        if n_classes >= k:
            results[f'top{k}_acc'] = top_k_accuracy_score(
                all_labels, all_probs, k=k, labels=range(n_classes))

    # MRR
    ranks = []
    for i in range(len(all_labels)):
        sorted_idx = np.argsort(-all_probs[i])
        rank = np.where(sorted_idx == all_labels[i])[0][0] + 1
        ranks.append(1.0 / rank)
    results['mrr'] = np.mean(ranks)

    y_pred = np.argmax(all_probs, axis=1)
    results['macro_f1'] = f1_score(all_labels, y_pred, average='macro', zero_division=0)
    results['weighted_f1'] = f1_score(all_labels, y_pred, average='weighted', zero_division=0)

    return results


def train_pathway_model(model, train_loader, val_loader, n_classes,
                        epochs=30, lr=1e-3, device='cuda'):
    """Train a pathway prediction model."""
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', patience=5, factor=0.5)

    # Class weights for imbalanced data
    all_labels = []
    for _, _, labels in train_loader:
        all_labels.append(labels)
    all_labels = torch.cat(all_labels)
    class_counts = torch.bincount(all_labels, minlength=n_classes).float()
    class_weights = (class_counts.sum() / (n_classes * class_counts.clamp(min=1))).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    history = {'train_loss': [], 'val_top1': [], 'val_top3': [], 'val_mrr': []}
    best_mrr = 0
    best_state = None

    for epoch in range(epochs):
        # Train
        model.train()
        total_loss = 0
        for emb, feats, labels in train_loader:
            emb, feats, labels = emb.to(device), feats.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(emb, feats)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(labels)
        avg_loss = total_loss / len(train_loader.dataset)

        # Validate
        metrics = evaluate_pathway(model, val_loader, device, n_classes)

        history['train_loss'].append(avg_loss)
        history['val_top1'].append(metrics['top1_acc'])
        history['val_top3'].append(metrics['top3_acc'])
        history['val_mrr'].append(metrics['mrr'])

        scheduler.step(metrics['mrr'])

        if metrics['mrr'] > best_mrr:
            best_mrr = metrics['mrr']
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1:3d}: loss={avg_loss:.4f}, "
                  f"Top1={metrics['top1_acc']:.4f}, "
                  f"Top3={metrics['top3_acc']:.4f}, "
                  f"MRR={metrics['mrr']:.4f}")

    # Restore best model
    if best_state:
        model.load_state_dict(best_state)
    model = model.to(device)

    return model, history, best_mrr


print("Training functions defined.")

## 6. Run Experiments: Ablation Study

Train 3 models and compare:
1. **Two-Tower (Qwen3 + Structured)** — Full model
2. **Structured-Only MLP** — Two-Tower without Qwen3 (use_qwen=False)
3. **PathwayResNet** — ResNet-style structured-only baseline

In [ ]:
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
N_FEATURES = len(FEATURE_NAMES)  # 38

# Filter samples to only include patients with embeddings
train_filtered = [s for s in train_samples if s['subject_id'] in patient_embeddings]
test_filtered = [s for s in test_samples if s['subject_id'] in patient_embeddings]

print(f"Train samples (with embeddings): {len(train_filtered):,}")
print(f"Test samples (with embeddings): {len(test_filtered):,}")

if len(train_filtered) < 100 or len(test_filtered) < 30:
    print("WARNING: Too few samples with embeddings. Set RUN_ALL=True and re-run embedding extraction.")

# Normalize structured features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
train_feats = np.array([s['features'] for s in train_filtered])
scaler.fit(train_feats)

for s in train_filtered + test_filtered:
    s['features'] = scaler.transform([s['features']])[0].tolist()

# Create datasets
train_ds = PathwayDataset(train_filtered, patient_embeddings, EMBED_DIM)
test_ds = PathwayDataset(test_filtered, patient_embeddings, EMBED_DIM)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=512, shuffle=False)

# ============================================================
# Experiment 1: Two-Tower (Qwen3 + Structured)
# ============================================================
print("\n" + "="*60)
print("  [1/3] Two-Tower (Qwen3 + Structured)")
print("="*60)
model_tt = PathwayTwoTower(EMBED_DIM, N_FEATURES, N_CLASSES, use_qwen=True)
model_tt, hist_tt, mrr_tt = train_pathway_model(
    model_tt, train_loader, test_loader, N_CLASSES, epochs=30, lr=1e-3, device=device)
metrics_tt = evaluate_pathway(model_tt, test_loader, device, N_CLASSES)
print(f"  Final: Top1={metrics_tt['top1_acc']:.4f}, Top3={metrics_tt['top3_acc']:.4f}, MRR={metrics_tt['mrr']:.4f}")

# ============================================================
# Experiment 2: Structured-Only MLP
# ============================================================
print("\n" + "="*60)
print("  [2/3] Structured-Only MLP (no Qwen3)")
print("="*60)
model_no_q = PathwayTwoTower(EMBED_DIM, N_FEATURES, N_CLASSES, use_qwen=False)
model_no_q, hist_no_q, mrr_no_q = train_pathway_model(
    model_no_q, train_loader, test_loader, N_CLASSES, epochs=30, lr=1e-3, device=device)
metrics_no_q = evaluate_pathway(model_no_q, test_loader, device, N_CLASSES)
print(f"  Final: Top1={metrics_no_q['top1_acc']:.4f}, Top3={metrics_no_q['top3_acc']:.4f}, MRR={metrics_no_q['mrr']:.4f}")

# ============================================================
# Experiment 3: PathwayResNet
# ============================================================
print("\n" + "="*60)
print("  [3/3] PathwayResNet")
print("="*60)
model_resnet = PathwayResNet(N_FEATURES, N_CLASSES)
model_resnet, hist_resnet, mrr_resnet = train_pathway_model(
    model_resnet, train_loader, test_loader, N_CLASSES, epochs=30, lr=1e-3, device=device)
metrics_resnet = evaluate_pathway(model_resnet, test_loader, device, N_CLASSES)
print(f"  Final: Top1={metrics_resnet['top1_acc']:.4f}, Top3={metrics_resnet['top3_acc']:.4f}, MRR={metrics_resnet['mrr']:.4f}")

# ============================================================
# Plot training curves
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for name, hist, color in [
    ('Two-Tower', hist_tt, '#e74c3c'),
    ('No Qwen3', hist_no_q, '#3498db'),
    ('ResNet', hist_resnet, '#2ecc71'),
]:
    axes[0].plot(hist['train_loss'], label=name, color=color)
    axes[1].plot(hist['val_top1'], label=name, color=color)
    axes[2].plot(hist['val_mrr'], label=name, color=color)

axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Train Loss')
axes[0].set_title('Training Loss'); axes[0].legend()
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Top-1 Accuracy')
axes[1].set_title('Validation Top-1 Accuracy'); axes[1].legend()
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('MRR')
axes[2].set_title('Validation MRR'); axes[2].legend()
plt.tight_layout()
plt.show()

print(f"\nSUMMARY:")
print(f"  Two-Tower: Top1={metrics_tt['top1_acc']:.4f}, MRR={metrics_tt['mrr']:.4f}")
print(f"  No-Qwen3:  Top1={metrics_no_q['top1_acc']:.4f}, MRR={metrics_no_q['mrr']:.4f}")
print(f"  ResNet:    Top1={metrics_resnet['top1_acc']:.4f}, MRR={metrics_resnet['mrr']:.4f}")
delta = metrics_tt['mrr'] - metrics_no_q['mrr']
print(f"  Qwen3 MRR improvement: {delta:+.4f} ({'positive' if delta > 0 else 'negative'})")

## 7. Results Summary & Comparison with Baselines

In [ ]:
# ============================================================
# Build results table
# ============================================================
dl_results = []
for name, metrics in [
    ('Two-Tower (Qwen3)', metrics_tt),
    ('Structured-Only MLP', metrics_no_q),
    ('PathwayResNet', metrics_resnet),
]:
    dl_results.append({
        'model': name,
        'top1_acc': round(metrics['top1_acc'], 4),
        'top3_acc': round(metrics['top3_acc'], 4),
        'top5_acc': round(metrics['top5_acc'], 4),
        'mrr': round(metrics['mrr'], 4),
        'macro_f1': round(metrics['macro_f1'], 4),
        'weighted_f1': round(metrics['weighted_f1'], 4),
    })

dl_df = pd.DataFrame(dl_results)
print("=== Deep Learning Model Results ===")
print(dl_df.to_string(index=False))

# Compare with baselines
if baseline_results:
    print("\n\n=== Full Comparison: All Models ===")
    print(f"{'Model':30s} {'Top-1':>8s} {'Top-3':>8s} {'Top-5':>8s} {'MRR':>8s}")
    print("-" * 66)

    # Baseline models
    for mname, mdata in baseline_results.get('models', {}).items():
        print(f"{mname:30s} {mdata['top1_acc']:8.4f} {mdata['top3_acc']:8.4f} "
              f"{mdata['top5_acc']:8.4f} {mdata['mrr']:8.4f}")

    print("-" * 66)

    # DL models
    for _, row in dl_df.iterrows():
        print(f"{row['model']:30s} {row['top1_acc']:8.4f} {row['top3_acc']:8.4f} "
              f"{row['top5_acc']:8.4f} {row['mrr']:8.4f}")

    # Best baseline vs Two-Tower
    best_baseline_mrr = max(m['mrr'] for m in baseline_results.get('models', {}).values())
    tt_mrr = metrics_tt['mrr']
    print(f"\nBest Baseline MRR: {best_baseline_mrr:.4f}")
    print(f"Two-Tower MRR:     {tt_mrr:.4f}")
    print(f"Improvement:       {tt_mrr - best_baseline_mrr:+.4f}")

# Save results
dl_df.to_csv(os.path.join(DATA_DIR, 'qwen3_pathway_results.csv'),
             index=False, encoding='utf-8-sig')
print(f"\nResults saved to {DATA_DIR}/qwen3_pathway_results.csv")

## 8. Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# Fig 1: Ablation Study bar chart
# ============================================================
fig, ax = plt.subplots(figsize=(12, 6))
metrics_list = ['top1_acc', 'top3_acc', 'top5_acc', 'mrr']
metric_labels = ['Top-1 Acc', 'Top-3 Acc', 'Top-5 Acc', 'MRR']
model_names = dl_df['model'].tolist()
x = np.arange(len(metrics_list))
width = 0.25
colors = ['#e74c3c', '#3498db', '#2ecc71']

for i, (_, row) in enumerate(dl_df.iterrows()):
    vals = [row[m] for m in metrics_list]
    ax.bar(x + i * width, vals, width, label=row['model'], color=colors[i], alpha=0.85)

ax.set_xticks(x + width)
ax.set_xticklabels(metric_labels)
ax.set_ylabel('Score')
ax.set_title('Ablation Study: Pathway Prediction Models', fontweight='bold')
ax.legend()
ax.set_ylim(0, 1.0)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================================
# Fig 2: Full model comparison (baselines + DL)
# ============================================================
if baseline_results:
    all_models = []
    for mname, mdata in baseline_results.get('models', {}).items():
        all_models.append({'model': mname, **mdata})
    for _, row in dl_df.iterrows():
        all_models.append(row.to_dict())

    comp_df = pd.DataFrame(all_models)
    comp_df = comp_df.sort_values('mrr')

    fig, ax = plt.subplots(figsize=(12, max(6, len(comp_df) * 0.6)))
    colors_bar = ['#95a5a6'] * len(baseline_results.get('models', {})) + ['#e74c3c', '#3498db', '#2ecc71']
    ax.barh(comp_df['model'], comp_df['mrr'], color=colors_bar[:len(comp_df)], alpha=0.85)
    ax.set_xlabel('MRR (Mean Reciprocal Rank)')
    ax.set_title('Comorbidity Pathway Prediction \u2014 All Models Comparison', fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

# ============================================================
# Fig 3: Qwen3 improvement delta
# ============================================================
fig, ax = plt.subplots(figsize=(8, 4))
deltas = {
    'Top-1 Acc': metrics_tt['top1_acc'] - metrics_no_q['top1_acc'],
    'Top-3 Acc': metrics_tt['top3_acc'] - metrics_no_q['top3_acc'],
    'Top-5 Acc': metrics_tt['top5_acc'] - metrics_no_q['top5_acc'],
    'MRR': metrics_tt['mrr'] - metrics_no_q['mrr'],
}
colors_delta = ['#2ecc71' if v > 0 else '#e74c3c' for v in deltas.values()]
ax.bar(deltas.keys(), deltas.values(), color=colors_delta)
ax.axhline(y=0, color='black', linewidth=0.5)
ax.set_ylabel('Improvement (Two-Tower \u2212 Structured-Only)')
ax.set_title('Qwen3 Embedding Added Value for Pathway Prediction', fontweight='bold')
plt.tight_layout()
plt.show()

print("\nVisualization complete.")